<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.3_Praktikum_Dataset_Training_dan_Testing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum AI Modul 5.3: Dataset (Training dan Testing)
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Capaian Pembelajaran Praktikum:
1. Membuktikan secara empiris dampak bahaya kebocoran data (*Data Leakage*) terhadap distorsi metrik evaluasi model AI.
2. Mengonfigurasi dan membandingkan skema validasi silang: *Stratified K-Fold* (data imbalanced), *TimeSeriesSplit* (data runtun waktu musiman), dan *GroupKFold* (data dengan autokorelasi spasial blok kebun).
3. Mengenkapsulasi alur pra-pemrosesan dan pemodelan ke dalam *Scikit-Learn Pipeline* bebas bocor.
4. Melakukan audit forensik terhadap repositori kode machine learning.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, TimeSeriesSplit, GroupKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, recall_score

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 1. Eksperimen 1: Pembuktian Bahaya Kebocoran Pra-Pemrosesan (*Preprocessing Leakage*)
Kita akan membandingkan dua pendekatan normalisasi data pada masalah estimasi produksi panen TBS:
- **Pendekatan A (Model Bocor / Leaky):** `StandardScaler.fit_transform(X)` dieksekusi pada seluruh dataset sebelum partisi train/test.
- **Pendekatan B (Model Bebas Bocor / Leak-Free):** Penskalaan diisolasi murni di dalam `Pipeline` sehingga parameter $\mu$ dan $\sigma$ hanya dipelajari dari $X_{\text{train}}$.


In [ ]:
# 1. Bangun dataset sintetis 100 sampel kebun
np.random.seed(42)
n_data = 100

x_fitur = np.random.uniform(50, 250, n_data)
# Sisipkan pencilan ekstrem pada 5 sampel terakhir untuk menguji ketahanan isolasi
x_fitur[-5:] = x_fitur[-5:] * 4.0
y_panen = 0.05 * x_fitur + np.random.normal(0, 1.5, n_data)

X_mat = x_fitur.reshape(-1, 1)

# MODEL A (BOCOR): fit_transform pada seluruh data sebelum split!
scaler_bocor = StandardScaler()
X_bocor = scaler_bocor.fit_transform(X_mat)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_bocor, y_panen, test_size=0.3, random_state=42)

reg_bocor = Ridge()
reg_bocor.fit(X_tr_b, y_tr_b)
r2_bocor = reg_bocor.score(X_te_b, y_te_b)

# MODEL B (BEBAS BOCOR): Split terlebih dahulu, fit HANYA pada X_train!
X_tr_murni, X_te_murni, y_tr_m, y_te_m = train_test_split(X_mat, y_panen, test_size=0.3, random_state=42)

pipe_murni = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge())
])
pipe_murni.fit(X_tr_murni, y_tr_m)
r2_murni = pipe_murni.score(X_te_murni, y_te_m)

print("=== HASIL KOMPARASI KEBOCORAN PRA-PEMROSESAN ===")
print(f"R2 Score Model Bocor (Terkontaminasi Outlier Uji): {r2_bocor:.4f}")
print(f"R2 Score Model Bebas Bocor (Evaluasi Obyektif Murni): {r2_murni:.4f}")
print(f"Mean yang dipelajari Model Bocor: {scaler_bocor.mean_[0]:.2f}")
print(f"Mean sejati Data Latih Model Murni: {pipe_murni.named_steps['scaler'].mean_[0]:.2f}")


## 2. Eksperimen 2: Stratified K-Fold pada Kasus Ketidakseimbangan Kelas Ekstrem
Pada deteksi infeksi jamur *Ganoderma*, sampel pohon sakit sangat langka (misal hanya $5\%$).
Kita akan membandingkan pembagian acak biasa (*Random K-Fold*) dengan *Stratified K-Fold* untuk melihat stabilitas sebaran kelas minoritas di setiap lipatan (*fold*).


In [ ]:
# 1. Dataset imbalanced: 200 pohon (190 sehat = 0, 10 sakit Ganoderma = 1)
y_imbalanced = np.array([0]*190 + [1]*10)
X_dummy = np.random.randn(200, 3)

# Bandingkan sebaran fold K-Fold Biasa vs Stratified K-Fold
k_biasa = KFold(n_splits=5, shuffle=True, random_state=12)
k_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)

print("=== DISTRIBUSI KELAS MINORITAS PER LIPATAN (FOLD) ===")
print("A. K-Fold Biasa (Random Split):")
for fold, (tr_idx, val_idx) in enumerate(k_biasa.split(X_dummy, y_imbalanced)):
    minoritas_val = y_imbalanced[val_idx].sum()
    print(f"   Fold {fold+1}: Sampel Ganoderma di Data Validasi = {minoritas_val} pohon")

print("\nB. Stratified K-Fold (Proporsional Terkendali):")
for fold, (tr_idx, val_idx) in enumerate(k_strat.split(X_dummy, y_imbalanced)):
    minoritas_val = y_imbalanced[val_idx].sum()
    print(f"   Fold {fold+1}: Sampel Ganoderma di Data Validasi = {minoritas_val} pohon")

print("\nKesimpulan: Stratified K-Fold melenyapkan risiko Zero-Shot Fold Trap!")


## 3. Eksperimen 3: Partisi Data Runtun Waktu Musiman (TimeSeriesSplit vs K-Fold Biasa)
Data produksi TBS bulanan memiliki dependensi kronologis (tren dan musiman).
Menggunakan K-Fold acak akan membocorkan data masa depan untuk memprediksi masa lalu. Kita buktikan menggunakan `TimeSeriesSplit`.


In [ ]:
# 1. Dataset Runtun Waktu 36 Bulan (3 Tahun Operasional Kebun)
bulan = np.arange(1, 37)
# Tren meningkat + pola musiman siklus 12 bulan
tren_panen = 100 + 2.5 * bulan + 20 * np.sin(2 * np.pi * bulan / 12) + np.random.normal(0, 3, 36)

X_time = bulan.reshape(-1, 1)
y_time = tren_panen

# Bandingkan TimeSeriesSplit (Forward Chaining)
tscv = TimeSeriesSplit(n_splits=3)

print("=== SKEMA VALIDASI TIME-SERIES SPLIT (FORWARD CHAINING) ===")
for split_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_time)):
    bulan_tr = bulan[tr_idx]
    bulan_val = bulan[val_idx]
    print(f"Split {split_idx+1}:")
    print(f"  - Data Latih   : Bulan ke-{bulan_tr.min()} s.d. Bulan ke-{bulan_tr.max()} ({len(bulan_tr)} bulan)")
    print(f"  - Data Validasi: Bulan ke-{bulan_val.min()} s.d. Bulan ke-{bulan_val.max()} ({len(bulan_val)} bulan)")


## 4. Eksperimen 4: Mengatasi Autokorelasi Spasial Menggunakan GroupKFold
Dua pohon di dalam satu blok kebun afdeling yang sama memiliki kemiripan sifat tanah dan iklim.
Jika sampel pohon dari satu blok terpecah ke train dan test, model hanya menghafal karakteristik blok tersebut.
`GroupKFold` menjamin bahwa seluruh sampel dari satu blok geografis berada utuh di satu fold!


In [ ]:
# 1. Dataset 300 pohon dari 6 Blok Afdeling (50 pohon per blok)
kode_blok = np.repeat([f'BLOK_{chr(65+i)}' for i in range(6)], 50)
X_spasial = np.random.randn(300, 4)
y_spasial = np.random.randint(0, 2, 300)

gkf = GroupKFold(n_splits=3)

print("=== SKEMA GROUP K-FOLD (ISOLASI BLOK GEOGRAFIS) ===")
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_spasial, y_spasial, groups=kode_blok)):
    blok_latih = np.unique(kode_blok[tr_idx])
    blok_uji = np.unique(kode_blok[val_idx])
    print(f"Fold {fold+1}:")
    print(f"  - Blok pada Data Latih   : {list(blok_latih)}")
    print(f"  - Blok pada Data Validasi: {list(blok_uji)}")
    print(f"  - Irisan Blok (Kontaminasi): {set(blok_latih).intersection(set(blok_uji))}")

print("\nKesimpulan: Tidak ada satupun blok kebun yang tumpang tindih antara data latih dan data validasi!")


## 5. Tugas Tantangan Eksplorasi Mandiri (HOTS Challenge)

### Tantangan: Audit Forensik Kebocoran Target (Target Leakage)
Diberikan dataset produksi kelapa sawit berikut yang memuat fitur mencurigakan: `['Usia_Tanaman', 'Curah_Hujan', 'Dosis_NPK', 'Total_Uang_Borongan_Panen']`.
Identifikasi fitur mana yang memicu *target leakage*, buktikan secara komparatif perbedaannya, dan tuliskan kode evaluasi yang benar!

### Solusi Tantangan:


In [ ]:
# Simulasi kasus audit forensik
np.random.seed(99)
n_sampel = 200

usia = np.random.randint(5, 25, n_sampel)
hujan = np.random.uniform(150, 300, n_sampel)
npk = np.random.uniform(100, 400, n_sampel)

# Target aktual: Tonase panen
tonase_target = 0.04 * hujan + 0.3 * usia + 0.02 * npk + np.random.normal(0, 1.5, n_sampel)

# Fitur bocor: Uang borongan panen dihitung per ton hasil panen (FITUR PASCA-KEJADIAN!)
upah_borongan = tonase_target * 125000 + np.random.normal(0, 500, n_sampel)

df_audit = pd.DataFrame({
    'Usia': usia,
    'Hujan': hujan,
    'NPK': npk,
    'Upah_Borongan': upah_borongan,
    'Tonase': tonase_target
})

# Model 1 (Dengan Fitur Bocor):
X_bocor = df_audit[['Usia', 'Hujan', 'NPK', 'Upah_Borongan']]
y = df_audit['Tonase']
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_bocor, y, test_size=0.3, random_state=42)

mod_bocor = Ridge().fit(X_tr_b, y_tr_b)
r2_bocor = mod_bocor.score(X_te_b, y_te_b)

# Model 2 (Pembersihan Target Leakage):
X_bersih = df_audit[['Usia', 'Hujan', 'NPK']] # Upah_Borongan DIBUANG!
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_bersih, y, test_size=0.3, random_state=42)

mod_bersih = Ridge().fit(X_tr_c, y_tr_c)
r2_bersih = mod_bersih.score(X_te_c, y_te_c)

print("=== HASIL AUDIT FORENSIK TARGET LEAKAGE ===")
print(f"R2 Model Terkontaminasi Upah Borongan : {r2_bocor:.4f}  <-- Ilusi Akurasi Palsu 99.9%!")
print(f"R2 Model Bersih Bebas Target Leakage: {r2_bersih:.4f}  <-- Kinerja Prediktif Nyata")
print("\nRekomendasi Auditor: Kolom 'Upah_Borongan' WAJIB DIBUANG dari matriks fitur!")
